# Architecture and task validation

This is an interface-validation report, not a model-selection benchmark.
Small training budgets exercise public methods across the registry. Metrics
are labeled by task and evaluation split; poor scores are not teaching targets.
For fitted teaching examples, use the [tutorials](../../tutorials/index.rst).


**Execution environment:** nampy 0.3.0, numpy 1.26.4, scikit-learn 1.9.0, torch 2.14.1+cpu, lightning 2.6.6. Python 3.12.13; source base commit `4cffdeb` (working checkout); elapsed 60.5 seconds on this CPU. Runtime varies by hardware.

## 1. Data and model setup

The same synthetic covariates are used throughout so differences in the
returned predictions, scores, and additive components reflect the modeled
quantity rather than an unrelated change of dataset.


In [1]:
from pathlib import Path
import tempfile
import time
import logging
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)
plt.style.use("seaborn-v0_8-whitegrid")
COLORS = ["#2563EB", "#F97316", "#10B981", "#8B5CF6", "#EF4444"]

N_SAMPLES = 1000
MAX_EPOCHS = 20
RANDOM_STATE = 7
CHECKPOINT_ROOT = Path(tempfile.mkdtemp(prefix="nampy-notebook-"))

NEURAL_FIT_KWARGS = {
    "max_epochs": MAX_EPOCHS,
    "batch_size": 256,
    "random_state": RANDOM_STATE,
    "checkpoint_path": CHECKPOINT_ROOT,
    "logger": False,
    "enable_progress_bar": False,
    "enable_model_summary": False,
    "num_sanity_val_steps": 0,
}

rng = np.random.default_rng(RANDOM_STATE)

from sklearn.metrics import accuracy_score, log_loss, roc_auc_score
from nampy import models
from nampy.neural.registry import architectures

n = N_SAMPLES
x = np.linspace(0.03, 0.97, n)
groups = pd.Categorical(np.resize(np.array(["a", "b", "c"]), n))
X = pd.DataFrame({"x1": x, "x2": np.cos(2 * np.pi * x), "group": groups})
y_reg = np.sin(2 * np.pi * x) + 0.25 * x + 0.2 * (groups == "b")
y_cls = np.resize(np.array([0, 1]), n)
y_lss = y_reg + np.exp(-1.2 + 0.8 * x) * rng.normal(size=n)

SWEEP_FIT_KWARGS = {
    **NEURAL_FIT_KWARGS,
    "batch_size": 256,
    "shuffle": False,
}

MODEL_KWARGS = {
    "linreg": {},
    "nam": {"layer_sizes": [8], "dropout": 0.0},
    "sian": {"layer_sizes": [8], "interactions": [("x1", "x2")]},
    "snam": {"layer_sizes": [8], "dropout": 0.0, "group_lasso_lambda": 0.01},
    "gpnam": {"rff_num_feat": 8},
    "igann": {"n_hid": 4, "n_estimators": 3, "early_stopping": 2},
    "nbm": {"layer_sizes": [8], "num_bases": 4, "dropout": 0.0, "bases_dropout": 0.0, "output_penalty": 0.01},
    "nbm_spam": {"layer_sizes": [8], "num_bases": 4, "ranks": [4], "batch_norm": False},
    "spam": {"ranks": [4], "dropout": 0.0},
    "natt": {"d_model": 8, "n_layers": 1, "n_heads": 2, "transformer_dim_feedforward": 16, "head_layer_sizes": (), "attn_dropout": 0.0},
    "namformer": {"d_model": 8, "n_layers": 1, "n_heads": 2, "transformer_dim_feedforward": 16, "head_layer_sizes": (), "attn_dropout": 0.0},
    "treenam": {"tree_depth": 2, "tree_lamda": 0.01},
    "ensemble_treenam": {"tree_depth": 2, "tree_lamda": 0.01, "num_estimators": 2},
    "nodegam": {"num_trees": 4, "num_layers": 1, "depth": 2, "last_dropout": 0.0, "anneal_steps": 10, "interaction_degree": 1},
    "qnam": {"layer_sizes": [8], "dropout": 0.0, "distributional_kwargs": {"quantiles": [0.1, 0.5, 0.9]}},
    "spline_nam": {"n_knots": 5, "smoothing": 0.01},
}

def estimator_class(spec, suffix):
    return getattr(models, f"{spec.estimator_prefix}{suffix}", None)

def make_estimator(name, spec, suffix, **extra):
    cls = estimator_class(spec, suffix)
    if cls is None:
        return None
    kwargs = dict(MODEL_KWARGS[name])
    kwargs.update(extra)
    return cls(**kwargs)

def reconstruction_error(estimator, features):
    components = estimator.predict_components(features, center=True)
    link = np.asarray(components.link)
    def aligned(value):
        array = np.asarray(value, dtype=float)
        if link.ndim == 1 and array.shape == (link.shape[0], 1):
            return array[:, 0]
        if link.ndim == 2 and array.shape == (link.shape[0],):
            return array[:, None]
        return array
    reconstruction = aligned(components.intercept)
    for contribution in components.terms.values():
        reconstruction = reconstruction + aligned(contribution)
    if components.offset is not None:
        reconstruction = reconstruction + aligned(components.offset)
    error = float(np.max(np.abs(reconstruction - link)))
    valid = error <= 1e-5
    assert valid, (type(estimator).__name__, error)
    return error, valid


## 2. Training-data regression checks

Every registered regression architecture is fitted on the same rows. The
score, runtime, additive reconstruction, and term importance are recorded.
Architecture-specific output is added where available. This confirms the
installed runtime rather than relying on a copied documentation matrix.

Under squared-error loss, the optimal point predictor is the conditional
mean $\mathbb E[Y\mid x]$. A good mean prediction does not describe how
widely outcomes vary around it. Read the regression scores as measures of
point prediction; later sections add likelihood and interval checks for the
rest of the conditional distribution.


In [2]:
regression_models = {}
regression_rows = []
for name, spec in architectures().items():
    if not spec.supports("regression"):
        continue
    estimator = make_estimator(name, spec, "Regressor")
    started = time.perf_counter()
    estimator.fit(X, y_reg, **SWEEP_FIT_KWARGS)
    importance = estimator.term_importance(X, center=True)
    error, reconstruction_valid = reconstruction_error(estimator, X)
    row = {
        "architecture": name,
        "status": "Demonstrated",
        "R2": estimator.score(X, y_reg),
        "seconds": time.perf_counter() - started,
        "reconstruction error": error,
        "reconstruction valid": reconstruction_valid,
        "most important term": importance.iloc[0]["term"] if len(importance) else None,
    }
    if hasattr(estimator, "local_term_importance") and spec.supports("local_term_importance"):
        row["special output"] = "local importance"
    elif name == "igann":
        row["special output"] = f"{estimator.n_estimators_} boosting stages"
    else:
        row["special output"] = None
    regression_models[name] = estimator
    regression_rows.append(row)

regression_report = pd.DataFrame(regression_rows)
regression_report


GPU available: False, used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


`Trainer.fit` stopped: `max_epochs=20` reached.


GPU available: False, used: False


...

GPU available: False, used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


`Trainer.fit` stopped: `max_epochs=20` reached.


,architecture,status,R2,seconds,reconstruction error,reconstruction valid,most important term,special output
0,linreg,Demonstrated,-0.416730,0.672396,1.150183e-07,True,group,NaN
1,nam,Demonstrated,-0.703698,0.749126,2.421439e-08,True,x1,NaN
2,sian,Demonstrated,0.489438,0.590850,7.264316e-08,True,x1:x2,NaN
3,snam,Demonstrated,-0.319681,0.877644,2.747402e-08,True,group,NaN
4,gpnam,Demonstrated,0.024490,0.068586,2.043089e-08,True,group[1],NaN
5,igann,Demonstrated,0.324776,0.076182,3.911555e-08,True,group,3 boosting stages
6,nbm,Demonstrated,-0.095009,0.817757,1.396984e-08,True,group[1],NaN
7,nbm_spam,Demonstrated,-0.004501,1.048789,1.629815e-08,True,group[0],NaN
8,spam,Demonstrated,-0.177864,0.709804,5.541369e-08,True,group[0],local importance
9,natt,Demonstrated,0.819874,1.302188,7.380731e-08,True,x1,NaN


## 3. Training-data classification checks

Binary classifier components reconstruct logits before the probability
transform. Multiclass prediction is supported where advertised, but neural
classifier component extraction is currently restricted to binary classification. Log loss evaluates probability quality,
ROC-AUC evaluates ranking for the positive class, accuracy evaluates hard
labels, and calibration error compares binned predicted and empirical
frequencies. Link-scale terms explain the model score—not an additive
decomposition of probability.

For binary labels, $p(x)=\operatorname{sigmoid}\{\eta(x)\}$ and the
negative log likelihood is

$$-\frac1n\sum_i\left[y_i\log p_i+(1-y_i)\log(1-p_i)\right].$$

Log loss penalizes confident wrong probabilities. AUC depends on ordering,
so a model can rank well while reporting poorly calibrated probabilities.
Compare the probability metrics together rather than interpreting the
classification score as a complete assessment.


In [3]:
def expected_calibration_error(y_true, probability, bins=6):
    edges = np.linspace(0, 1, bins + 1)
    index = np.clip(np.digitize(probability, edges) - 1, 0, bins - 1)
    total = len(y_true)
    return sum(
        np.sum(index == b) / total * abs(np.mean(probability[index == b]) - np.mean(y_true[index == b]))
        for b in range(bins) if np.any(index == b)
    )

classification_models = {}
classification_rows = []
for name, spec in architectures().items():
    if not spec.supports("classification"):
        continue
    estimator = make_estimator(name, spec, "Classifier")
    started = time.perf_counter()
    estimator.fit(X, y_cls, **SWEEP_FIT_KWARGS)
    probability = estimator.predict_proba(X)[:, 1]
    label = estimator.predict(X)
    importance = estimator.term_importance(X, center=True)
    error, reconstruction_valid = reconstruction_error(estimator, X)
    classification_rows.append({
        "architecture": name, "status": "Demonstrated",
        "log loss": log_loss(y_cls, probability),
        "ROC-AUC": roc_auc_score(y_cls, probability),
        "accuracy": accuracy_score(y_cls, label),
        "calibration error": expected_calibration_error(y_cls, probability),
        "reconstruction error": error,
        "reconstruction valid": reconstruction_valid,
        "most important term": importance.iloc[0]["term"] if len(importance) else None,
        "seconds": time.perf_counter() - started,
    })
    classification_models[name] = estimator
classification_report = pd.DataFrame(classification_rows)
classification_report


GPU available: False, used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


`Trainer.fit` stopped: `max_epochs=20` reached.


GPU available: False, used: False


...

GPU available: False, used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


`Trainer.fit` stopped: `max_epochs=20` reached.


,architecture,status,log loss,ROC-AUC,accuracy,calibration error,reconstruction error,reconstruction valid,most important term,seconds
0,linreg,Demonstrated,0.720785,0.500800,0.500,0.096465,1.545995e-07,True,group,0.989068
1,nam,Demonstrated,0.703513,0.500876,0.500,0.065429,3.213063e-08,True,x1,0.825984
2,sian,Demonstrated,0.694319,0.498836,0.500,0.018074,3.259629e-08,True,x1:x2,0.648545
3,snam,Demonstrated,0.703215,0.500716,0.499,0.063022,2.793968e-08,True,group,0.930691
4,gpnam,Demonstrated,0.693391,0.500634,0.507,0.002121,1.536682e-08,True,group[0],0.754892
5,igann,Demonstrated,0.693474,0.501000,0.501,0.011553,3.725290e-09,True,group,0.081967
6,nbm,Demonstrated,0.697058,0.499310,0.500,0.038224,2.235174e-08,True,group[1],0.935589
7,nbm_spam,Demonstrated,0.693180,0.500406,0.500,0.003792,1.443550e-08,True,group[0],0.959199
8,spam,Demonstrated,0.695024,0.501568,0.500,0.026432,5.215406e-08,True,x1,0.802765
9,natt,Demonstrated,0.696564,0.500672,0.501,0.037526,1.565786e-08,True,group,1.525821


## 4. Training-data location-scale checks

Normal LSS models predict a location and positive scale. Each parameter
has its own link-scale additive predictor, so raw component reconstruction
is checked column by column. Mean negative log likelihood evaluates the
complete conditional distribution; interval coverage checks whether the
predicted scale is calibrated. QNAM is intentionally handled in the
quantile gallery because its fixed family is not normal.

A normal location-scale model has
$Y\mid x\sim\mathcal N\{\mu(x),\sigma(x)^2\}$, with separate additive
predictors transformed to a real location and positive scale. Its negative
log likelihood includes both $\log\sigma$ and the squared residual divided
by $2\sigma^2$. Thus scale must explain the size of residual variation,
not simply become large everywhere.

For a fitted normal distribution, $\mu\pm1.96\sigma$ is a nominal 95%
prediction interval. It describes future outcomes conditional on the fitted
parameters, not a confidence interval for the mean. Held-out coverage and
interval width help reveal whether the predicted spread is useful.


In [4]:
lss_models = {}
lss_rows = []
for name, spec in architectures().items():
    if not spec.supports("distributional") or name == "qnam":
        continue
    estimator = make_estimator(name, spec, "LSS", family="normal")
    started = time.perf_counter()
    estimator.fit(X, y_lss, **SWEEP_FIT_KWARGS)
    parameters = estimator.predict(X)
    raw = estimator.predict(X, raw=True)
    lower = parameters[:, 0] - 1.645 * parameters[:, 1]
    upper = parameters[:, 0] + 1.645 * parameters[:, 1]
    error, reconstruction_valid = reconstruction_error(estimator, X)
    lss_rows.append({
        "architecture": name, "status": "Demonstrated",
        "NLL": estimator.evaluate(X, y_lss)["NLL"],
        "minimum scale": parameters[:, 1].min(),
        "90% coverage": np.mean((y_lss >= lower) & (y_lss <= upper)),
        "raw width": raw.shape[1],
        "reconstruction error": error,
        "reconstruction valid": reconstruction_valid,
        "seconds": time.perf_counter() - started,
    })
    lss_models[name] = estimator
lss_report = pd.DataFrame(lss_rows)
lss_report


GPU available: False, used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


`Trainer.fit` stopped: `max_epochs=20` reached.


GPU available: False, used: False


...

GPU available: False, used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


`Trainer.fit` stopped: `max_epochs=20` reached.


,architecture,status,NLL,minimum scale,90% coverage,raw width,reconstruction error,reconstruction valid,seconds
0,linreg,Demonstrated,1.298374,0.491920,0.916,2,1.150183e-07,True,0.747015
1,nam,Demonstrated,1.351217,0.526378,0.784,2,3.911555e-08,True,0.676562
2,sian,Demonstrated,0.985271,0.530487,0.922,2,7.078052e-08,True,0.643013
3,snam,Demonstrated,1.248810,0.787541,0.921,2,5.261973e-08,True,0.879417
4,gpnam,Demonstrated,1.142782,0.496734,0.922,2,8.940697e-08,True,0.604548
5,igann,Demonstrated,0.898148,0.438979,0.903,2,1.890585e-07,True,0.620174
6,nbm,Demonstrated,1.222517,0.683150,0.869,2,2.072193e-08,True,0.748315
7,nbm_spam,Demonstrated,1.185842,0.701436,0.868,2,4.703179e-08,True,0.935689
8,spam,Demonstrated,1.295459,0.629661,0.837,2,7.590279e-08,True,0.628839
9,natt,Demonstrated,1.008232,0.431293,0.940,2,7.031485e-08,True,1.229512


## 5. Distribution families

A distribution family owns target support, output width, parameter
transforms, likelihood, point prediction, and default metrics. Continuous,
count, discrete/ordered, multivariate, and quantile families therefore
require different targets. The loop fits one small canonical NAMLSS model
per installed family. These fits test the interface and output domains;
they are not estimates worth interpreting scientifically.

Distributional regression specifies
$p\{y\mid\theta_1(x),\ldots,\theta_K(x)\}$ rather than only a mean.
Each parameter has a domain enforced by its transform. Counts, positive
continuous responses, and bounded outcomes call for different likelihoods;
a valid transform alone does not make a family suitable for the observed
data. Compare likelihood scores only when they refer to the same target
and a compatible probability measure.


In [5]:
from nampy.models import NAMLSS
from nampy.neural.distributions.registry import FAMILY_REGISTRY

positive = np.exp(0.2 + 0.4 * x + rng.normal(0, 0.15, n))
counts = rng.poisson(np.exp(0.2 + 0.5 * x))
zero_counts = counts.copy(); zero_counts[::3] = 0
classes3 = np.resize(np.arange(3), n)
composition = rng.dirichlet([2.0, 3.0, 4.0], size=n)
multi_y = np.column_stack([y_lss, np.cos(np.pi * x) + rng.normal(0, 0.1, n)])

family_targets = {
    "normal": y_lss, "poisson": counts, "gamma": positive,
    "beta": np.clip(1 / (1 + np.exp(-y_lss)), 1e-4, 1-1e-4),
    "dirichlet": composition, "studentt": y_lss,
    "negativebinom": counts, "inversegamma": positive,
    "categorical": classes3, "quantile": y_lss, "robustnormal": y_lss,
    "lognormal": positive, "weibull": positive, "loglogistic": positive,
    "zip": zero_counts, "zinb": zero_counts,
    "hurdlepoisson": zero_counts, "hurdlenegativebinom": zero_counts,
    "tweedie": zero_counts.astype(float), "ordinal": classes3,
    "mvnormdiag": multi_y,
}
family_groups = {
    "continuous": {"normal", "gamma", "beta", "studentt", "inversegamma", "robustnormal", "lognormal", "weibull", "loglogistic", "tweedie"},
    "counts": {"poisson", "negativebinom", "zip", "zinb", "hurdlepoisson", "hurdlenegativebinom"},
    "discrete and ordered": {"categorical", "ordinal"},
    "multivariate": {"dirichlet", "mvnormdiag"},
    "quantiles": {"quantile"},
}

def group_for_family(name):
    return next(group for group, names in family_groups.items() if name in names)

family_models = {}
family_rows = []
for name in FAMILY_REGISTRY:
    kwargs = {"layer_sizes": [8], "dropout": 0.0}
    if name == "quantile":
        kwargs["distributional_kwargs"] = {"quantiles": [0.1, 0.5, 0.9]}
    estimator = NAMLSS(family=name, **kwargs)
    target = family_targets[name]
    started = time.perf_counter()
    estimator.fit(X, target, **SWEEP_FIT_KWARGS)
    raw = estimator.predict(X, raw=True)
    natural = estimator.predict(X)
    metrics = estimator.evaluate(X, target)
    family_rows.append({
        "group": group_for_family(name), "family": name,
        "status": "Demonstrated", "target shape": np.asarray(target).shape,
        "raw parameters": raw.shape[1], "finite output": bool(np.isfinite(natural).all()),
        "metric": next(iter(metrics)), "metric value": next(iter(metrics.values())),
        "seconds": time.perf_counter() - started,
    })
    family_models[name] = estimator
family_report = pd.DataFrame(family_rows).sort_values(["group", "family"])
family_report


GPU available: False, used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


`Trainer.fit` stopped: `max_epochs=20` reached.


GPU available: False, used: False


...

GPU available: False, used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


`Trainer.fit` stopped: `max_epochs=20` reached.


,group,family,status,target shape,raw parameters,finite output,metric,metric value,seconds
3,continuous,beta,Demonstrated,"(1000,)",2,True,NLL,0.175786,0.676694
2,continuous,gamma,Demonstrated,"(1000,)",2,True,NLL,1.575774,0.768915
7,continuous,inversegamma,Demonstrated,"(1000,)",2,True,NLL,1.659127,0.652896
13,continuous,loglogistic,Demonstrated,"(1000,)",2,True,NLL,2.220873,0.644770
11,continuous,lognormal,Demonstrated,"(1000,)",2,True,NLL,0.950339,0.774606
0,continuous,normal,Demonstrated,"(1000,)",2,True,NLL,1.351217,0.646971
10,continuous,robustnormal,Demonstrated,"(1000,)",2,True,NLL,0.433843,0.644150
5,continuous,studentt,Demonstrated,"(1000,)",3,True,NLL,1.703900,0.727738
18,continuous,tweedie,Demonstrated,"(1000,)",2,True,NLL,2.525686,0.735259
12,continuous,weibull,Demonstrated,"(1000,)",2,True,NLL,1.909803,0.674513


### 5.1 Inspect parameter constraints

Natural transforms enforce positive rates/scales and unit-interval
probabilities. They do not guarantee calibration. QNAM additionally
enforces ordered quantiles numerically.

For quantile outputs, the pinball loss weights positive and negative
residuals asymmetrically to target a chosen probability level. An ordered
parameterization ensures $q_{\tau_1}(x)\leq q_{\tau_2}(x)$ when
$\tau_1<\tau_2$. The parameter checks below verify output domains and
ordering. Calibration is a separate empirical property that needs held-out
outcomes.


In [6]:
domain_checks = {
    "normal scale positive": bool((family_models["normal"].predict(X)[:, 1] > 0).all()),
    "beta parameters positive": bool((family_models["beta"].predict(X) > 0).all()),
    "ZIP finite": bool(np.isfinite(family_models["zip"].predict(X)).all()),
}

qnam = make_estimator("qnam", architectures()["qnam"], "LSS")
qnam.fit(X, y_lss, **SWEEP_FIT_KWARGS)
quantiles = qnam.predict(X)
minimum_quantile_gap = float(np.diff(quantiles, axis=1).min())
domain_checks["QNAM minimum adjacent quantile gap"] = minimum_quantile_gap
domain_checks


GPU available: False, used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


`Trainer.fit` stopped: `max_epochs=20` reached.


{'normal scale positive': True,
 'beta parameters positive': True,
 'ZIP finite': True,
 'QNAM minimum adjacent quantile gap': 2.7959256172180176}

## 6. Ensemble uncertainty

`NeuralEnsemble` clones and fits regressors or classifiers independently.
Bootstrapping changes member datasets; random seeds change initialization
and data order; `n_jobs` controls parallel fitting. Predictions are
averaged on the response scale, while terms and links are averaged on the
additive scale. Persistence is inherited from the member estimators only;
the complete ensemble can be serialized with joblib.

For response predictions $\hat\mu_m(x)$, the ensemble prediction is
$\bar\mu(x)=M^{-1}\sum_m\hat\mu_m(x)$. Their sample standard deviation
measures disagreement among the fitted members. It does not include
observation noise unless that is modeled separately.

With a nonlinear inverse link, averaging response predictions is different
from transforming an average predictor:
$M^{-1}\sum_m g^{-1}(\eta_m)\ne g^{-1}(M^{-1}\sum_m\eta_m)$ in general.
The averaged components therefore explain the averaged link predictor;
do not automatically equate its inverse transform with the ensemble's
response-scale prediction.


In [7]:
from joblib import dump, load
from nampy.models import NAMRegressor, NeuralEnsemble

ensemble = NeuralEnsemble(
    NAMRegressor(layer_sizes=[8], dropout=0.0),
    n_estimators=3, random_state=RANDOM_STATE, n_jobs=1, bootstrap=True,
)
ensemble.fit(X, y_reg, **SWEEP_FIT_KWARGS)
ensemble_prediction = ensemble.predict(X)
ensemble_components = ensemble.predict_components(X, center=True)
uncertainty = ensemble.predict_component_uncertainty(X, center=True)
ensemble_reconstruction_error, _ = reconstruction_error(ensemble, X)

ensemble_path = CHECKPOINT_ROOT / "neural-ensemble.joblib"
dump(ensemble, ensemble_path)
restored_ensemble = load(ensemble_path)
reload_difference = float(np.max(np.abs(restored_ensemble.predict(X) - ensemble_prediction)))

display({
    "members": uncertainty.n_estimators,
    "mean response SD": float(np.mean(uncertainty.response_std)),
    "mean term SD": {name: float(values.mean()) for name, values in uncertainty.term_std.items()},
    "reconstruction error": ensemble_reconstruction_error,
    "maximum reload difference": reload_difference,
})


GPU available: False, used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


`Trainer.fit` stopped: `max_epochs=20` reached.


GPU available: False, used: False


...

GPU available: False, used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


`Trainer.fit` stopped: `max_epochs=20` reached.


{'members': 3,
 'mean response SD': 0.1983688920736313,
 'mean term SD': {'x1': 0.04003573446966001,
  'x2': 0.12111195366401764,
  'group': 0.034382466944564045},
 'reconstruction error': 3.89603276862438e-08,
 'maximum reload difference': 0.0}

### 6.1 Three different meanings of uncertainty

- **Ensemble disagreement** measures variation among fitted neural members
  under seeds and/or bootstrap samples. It is not automatically a
  calibrated confidence interval.
- **GAM standard errors** arise from an approximate sampling covariance for
  fitted coefficients, conditional on model assumptions and optionally
  including smoothing uncertainty.
- **LSS conditional uncertainty** is the predicted response distribution
  for a new outcome given its features. It can be large even if parameter
  estimates are known precisely.


## 7. Supported tasks and outputs

Cells use four states. **Demonstrated** means this notebook executed the
task successfully. **Supported but not demonstrated** comes from the
registry but lacked a successful run. **Unavailable** means the registry
does not expose the capability. **Not applicable** is reserved for a
concept that does not make sense for that architecture.


In [8]:
def task_status(name, capability, report):
    spec = architectures()[name]
    if not spec.supports(capability):
        return "Unavailable"
    match = report.loc[report["architecture"] == name]
    if len(match) and match.iloc[0]["status"] == "Demonstrated":
        return "Demonstrated"
    return "Supported but not demonstrated"

capability_rows = []
for name, spec in architectures().items():
    capability_rows.append({
        "model": name,
        "Regression": task_status(name, "regression", regression_report),
        "Classification": task_status(name, "classification", classification_report),
        "LSS": "Demonstrated" if name == "qnam" else task_status(name, "distributional", lss_report),
        "Main effects": "Demonstrated" if spec.supports("additive_components") else "Unavailable",
        "Pairwise interactions": "Supported but not demonstrated" if spec.supports("interactions") else "Unavailable",
        "Higher-order interactions": "Supported but not demonstrated" if name in {"sian", "nbm", "spam", "nbm_spam"} else "Not applicable",
        "Sparsity": "Supported but not demonstrated" if name in {"snam", "sian", "nbm", "igann"} else "Not applicable",
        "Local importance": "Demonstrated" if name == "spam" and name in regression_models else ("Unavailable" if not spec.supports("local_term_importance") else "Supported but not demonstrated"),
        "Quantiles": "Demonstrated" if name == "qnam" else "Unavailable",
    })

final_capability_report = pd.DataFrame(capability_rows)
final_capability_report


,model,Regression,Classification,LSS,Main effects,Pairwise interactions,Higher-order interactions,Sparsity,Local importance,Quantiles
0,linreg,Demonstrated,Demonstrated,Demonstrated,Demonstrated,Unavailable,Not applicable,Not applicable,Unavailable,Unavailable
1,nam,Demonstrated,Demonstrated,Demonstrated,Demonstrated,Supported but not demonstrated,Not applicable,Not applicable,Unavailable,Unavailable
2,sian,Demonstrated,Demonstrated,Demonstrated,Demonstrated,Supported but not demonstrated,Supported but not demonstrated,Supported but not demonstrated,Unavailable,Unavailable
3,snam,Demonstrated,Demonstrated,Demonstrated,Demonstrated,Supported but not demonstrated,Not applicable,Supported but not demonstrated,Unavailable,Unavailable
4,gpnam,Demonstrated,Demonstrated,Demonstrated,Demonstrated,Supported but not demonstrated,Not applicable,Not applicable,Unavailable,Unavailable
5,igann,Demonstrated,Demonstrated,Demonstrated,Demonstrated,Unavailable,Not applicable,Supported but not demonstrated,Unavailable,Unavailable
6,nbm,Demonstrated,Demonstrated,Demonstrated,Demonstrated,Supported but not demonstrated,Supported but not demonstrated,Supported but not demonstrated,Unavailable,Unavailable
7,nbm_spam,Demonstrated,Demonstrated,Demonstrated,Demonstrated,Supported but not demonstrated,Supported but not demonstrated,Not applicable,Unavailable,Unavailable
8,spam,Demonstrated,Demonstrated,Demonstrated,Demonstrated,Supported but not demonstrated,Supported but not demonstrated,Not applicable,Demonstrated,Unavailable
9,natt,Demonstrated,Demonstrated,Demonstrated,Demonstrated,Supported but not demonstrated,Not applicable,Not applicable,Unavailable,Unavailable


## 8. Interpreting the results

Successful execution verifies the estimator interface and output shapes;
it does not establish that an architecture is suitable for a particular
dataset. For a substantive comparison, choose metrics before fitting, tune
inside cross-validation, use repeated splits, inspect term stability, and
compare against simple baselines.
